In [2]:
# =========================================================
# NOTEBOOK: 16_rag_chatbot.ipynb
# =========================================================

# =========================================================
# STEP 1 — IMPORT LIBRARIES
# =========================================================

import pandas as pd
import numpy as np

from langchain_text_splitters import (
    RecursiveCharacterTextSplitter
)

from langchain_community.vectorstores import FAISS

from langchain_huggingface import (
    HuggingFaceEmbeddings
)

from langchain_ollama import OllamaLLM

from langchain_core.documents import Document

import warnings

warnings.filterwarnings('ignore')

print("Libraries Imported Successfully")


Libraries Imported Successfully


In [3]:
import langchain

print(langchain.__version__)

1.3.0


In [4]:
# =========================================================
# STEP 2 — LOAD DATASET
# =========================================================

master_df = pd.read_csv(
    r"C:\Users\niran\Desktop\AI_Ecommerce_Customer_Intelligence_Platform\data\processed\statistical_analysis_dataset.csv"
)

print(master_df.shape)

master_df.head()

(2530433, 87)


,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,delivery_duration_days,delivery_delay_days,...,Monetary,churn,product_revenue,product_total_orders,product_avg_review,product_avg_delay,seller_total_revenue,seller_total_orders,seller_avg_review,seller_avg_delay
0,720fd9e9-cdeb-4dec-a187-f71586eb085a,1e2e2881-a0eb-4cb0-829f-a566e810d05f,canceled,2025-12-27 07:07:20,2025-12-27 08:33:20,NaN,NaN,2026-01-04 08:33:20,0.037357,0.036951,...,-0.233903,0,1234480.37,724,3.800000,-8.556017,6655560.54,5121,3.952081,-9.193164
1,720fd9e9-cdeb-4dec-a187-f71586eb085a,1e2e2881-a0eb-4cb0-829f-a566e810d05f,canceled,2025-12-27 07:07:20,2025-12-27 08:33:20,NaN,NaN,2026-01-04 08:33:20,0.037357,0.036951,...,-0.233903,0,4597968.11,2719,3.779849,-16.312730,6574599.35,4999,3.996998,-18.445178
2,720fd9e9-cdeb-4dec-a187-f71586eb085a,1e2e2881-a0eb-4cb0-829f-a566e810d05f,canceled,2025-12-27 07:07:20,2025-12-27 08:33:20,NaN,NaN,2026-01-04 08:33:20,0.037357,0.036951,...,-0.233903,0,1801075.20,2017,4.059165,-3.545635,6634183.36,4946,3.950022,-18.226289
3,c0142972-63fa-4af2-8070-f583ab769847,380b7418-308c-4bf7-b2bd-3ee446cb9ea6,delivered,2019-06-07 19:30:44,2019-06-08 05:08:44,2019-06-09 05:08:44,2019-06-14 05:08:44,2019-06-16 05:08:44,0.033655,0.036951,...,-0.057950,0,1310370.47,771,3.807428,-2.662776,6668289.19,5145,3.983153,-5.763848
4,c0142972-63fa-4af2-8070-f583ab769847,380b7418-308c-4bf7-b2bd-3ee446cb9ea6,delivered,2019-06-07 19:30:44,2019-06-08 05:08:44,2019-06-09 05:08:44,2019-06-14 05:08:44,2019-06-16 05:08:44,0.033655,0.036951,...,-0.057950,0,952878.10,1034,4.154799,-21.628627,6576178.28,5033,3.997017,-12.790185


In [5]:
# =========================================================
# STEP 3 — SAMPLE DATA
# =========================================================

# Using sample for faster embedding generation

master_df = master_df.sample(
    n=20000,
    random_state=42
)

print(master_df.shape)

(20000, 87)


In [6]:
# =========================================================
# STEP 4 — SELECT IMPORTANT COLUMNS
# =========================================================

rag_df = master_df[[

    'customer_segment',

    'product_category_name',

    'product_brand',

    'payment_type',

    'review_score',

    'delivery_delay_days',

    'total_spent',

    'avg_review_score',

    'product_revenue',

    'seller_total_revenue',

    'seller_avg_review',

    'churn'
]]

rag_df = rag_df.fillna("Unknown")

rag_df.head()

,customer_segment,product_category_name,product_brand,payment_type,review_score,delivery_delay_days,total_spent,avg_review_score,product_revenue,seller_total_revenue,seller_avg_review,churn
193882,0,2,3,0,4.0,1.206678,0.106217,3.900000,7667416.47,6693999.91,3.995152,1
26723,1,2,7,1,3.0,0.036951,-0.706028,3.000000,5946404.44,6565697.83,4.001737,1
1413181,2,2,7,3,5.0,0.036951,1.557518,5.000000,4207318.71,6756926.81,4.021611,0
887824,0,2,1,5,2.0,0.036951,0.612102,3.038462,4765958.71,6373764.73,3.990228,0
2462839,2,2,4,0,5.0,0.036951,1.111942,3.142857,4596402.88,6405798.88,3.975048,0


In [7]:
# =========================================================
# STEP 5 — CONVERT ROWS TO TEXT
# =========================================================

documents = []

for _, row in rag_df.iterrows():

    text = f"""

    Customer Segment: {row['customer_segment']}

    Product Category: {row['product_category_name']}

    Product Brand: {row['product_brand']}

    Payment Type: {row['payment_type']}

    Review Score: {row['review_score']}

    Delivery Delay: {row['delivery_delay_days']}

    Total Spent: {row['total_spent']}

    Average Review Score: {row['avg_review_score']}

    Product Revenue: {row['product_revenue']}

    Seller Revenue: {row['seller_total_revenue']}

    Seller Average Review: {row['seller_avg_review']}

    Customer Churn: {row['churn']}
    """

    documents.append(
        Document(page_content=text)
    )

print("Documents Created:", len(documents))

Documents Created: 20000


In [8]:
# =========================================================
# STEP 6 — TEXT SPLITTING
# =========================================================

text_splitter = RecursiveCharacterTextSplitter(

    chunk_size=500,

    chunk_overlap=50
)

split_docs = text_splitter.split_documents(
    documents
)

print("Chunks Created:", len(split_docs))

Chunks Created: 20000


In [9]:
# =========================================================
# STEP 7 — EMBEDDING MODEL
# =========================================================

embedding_model = HuggingFaceEmbeddings(

    model_name='sentence-transformers/all-MiniLM-L6-v2'
)

print("Embedding Model Loaded")

Embedding Model Loaded


In [10]:
# =========================================================
# STEP 8 — CREATE VECTOR DATABASE
# =========================================================

vector_db = FAISS.from_documents(

    split_docs,

    embedding_model
)

print("FAISS Vector DB Created")

FAISS Vector DB Created


In [ ]:
# =========================================================
# STEP 9 — SAVE VECTOR DATABASE
# =========================================================

from src.utils.path_config import VECTOR_DB_DIR

vector_db.save_local(VECTOR_DB_DIR)

print("Vector DB Saved Successfully")

Vector DB Saved Successfully


In [12]:
# =========================================================
# STEP 10 — LOAD LLM
# =========================================================

llm = OllamaLLM(
    model='llama3'
)

print("LLM Loaded Successfully")

LLM Loaded Successfully


In [13]:
# =========================================================
# STEP 11 — CREATE RETRIEVER
# =========================================================

retriever = vector_db.as_retriever(

    search_kwargs={"k": 5}
)

print("Retriever Created")

Retriever Created


In [14]:
# =========================================================
# STEP 12 — TEST RAG QUESTION
# =========================================================

query = "Which products have poor customer reviews?"

docs = retriever.invoke(query)

context = "\n".join([
    doc.page_content for doc in docs
])

prompt = f"""

Answer the question using the context below.

Context:
{context}

Question:
{query}

Answer:
"""

response = llm.invoke(prompt)

print("\nQUESTION:\n")

print(query)

print("\nANSWER:\n")

print(response)


QUESTION:

Which products have poor customer reviews?

ANSWER:

Based on the provided data, products with poor customer reviews are those that have a Review Score of 1.0.

From the context, there are four customers with a review score of 1.0:

* Customer Segment: 2.0, Product Category: 3.0, Product Brand: 2.0
* Customer Segment: 2.0, Product Category: 3.0, Product Brand: 1.0
* Customer Segment: 2.0, Product Category: 2.0, Product Brand: 0.0
* Customer Segment: 2.0, Product Category: 1.0, Product Brand: 1.0


In [15]:
# =========================================================
# STEP 13 — MULTIPLE BUSINESS QUESTIONS
# =========================================================

questions = [

    "Which customer segments are likely to churn?",

    "What products have highest revenue?",

    "Which payment types are most common?",

    "Which sellers have low review scores?",

    "What factors affect customer satisfaction?"
]

for q in questions:

    docs = retriever.invoke(q)

    context = "\n".join([
        doc.page_content for doc in docs
    ])

    prompt = f"""

    Answer the question using the context below.

    Context:
    {context}

    Question:
    {q}

    Answer:
    """

    answer = llm.invoke(prompt)

    print("\n================================================")

    print("QUESTION:")

    print(q)

    print("\nANSWER:")

    print(answer)


QUESTION:
Which customer segments are likely to churn?

ANSWER:
Based on the given data, all three customer segments (Customer Segment: 1.0) have a high Customer Churn rate of 1.0, which means that it is likely that customers in these segments will churn.

QUESTION:
What products have highest revenue?

ANSWER:
Based on the data provided, it appears that all the customers in Segment 2.0 are purchasing products from Category 3.0 with Brand 1.0.

For these specific product combinations, the revenue is:

* Product Revenue: 1797971.53
* Seller Revenue: 6417507.41

These are the highest revenue values among all customer segments and product combinations.

QUESTION:
Which payment types are most common?

ANSWER:
Based on the provided data, it seems that there is only one unique payment type mentioned across all customer segments: 3.0. Therefore, this is the most common payment type.

QUESTION:
Which sellers have low review scores?

ANSWER:
Based on the provided data, the sellers with low revi

In [16]:
# =========================================================
# STEP 14 — SINGLE QUESTION TEST
# =========================================================

query = "Which products have poor customer reviews?"

docs = retriever.invoke(query)

context = "\n".join([
    doc.page_content for doc in docs
])

prompt = f"""

Answer the question using the context below.

Context:
{context}

Question:
{query}

Answer:
"""

answer = llm.invoke(prompt)

print("\nAI Assistant:\n")

print(answer)


AI Assistant:

Based on the data, the product with a review score of 1.0 has poor customer reviews.


In [17]:
# =========================================================
# STEP 15 — BUSINESS INSIGHTS
# =========================================================

print("""

===================================================

BUSINESS INSIGHTS

===================================================

1. Built GenAI-powered RAG chatbot for e-commerce analytics.

2. Combined:
   - Vector embeddings
   - FAISS retrieval
   - LLM question answering
   - Retrieval-Augmented Generation

3. Created AI assistant capable of:
   - Business intelligence
   - Customer analytics
   - Seller analysis
   - Revenue insights
   - Review intelligence

4. Used:
   - LangChain
   - FAISS
   - Sentence Transformers
   - Ollama
   - Llama3

5. System supports:
   - AI-powered analytics
   - Enterprise chatbot workflows
   - Semantic search
   - Context-aware question answering

===================================================

""")




BUSINESS INSIGHTS


1. Built GenAI-powered RAG chatbot for e-commerce analytics.

2. Combined:
   - Vector embeddings
   - FAISS retrieval
   - LLM question answering
   - Retrieval-Augmented Generation

3. Created AI assistant capable of:
   - Business intelligence
   - Customer analytics
   - Seller analysis
   - Revenue insights
   - Review intelligence

4. Used:
   - LangChain
   - FAISS
   - Sentence Transformers
   - Ollama
   - Llama3

5. System supports:
   - AI-powered analytics
   - Enterprise chatbot workflows
   - Semantic search
   - Context-aware question answering





In [18]:
# =========================================================
# STEP 16 — NOTEBOOK COMPLETION
# =========================================================

print("""

===================================================
16_rag_chatbot.ipynb COMPLETED SUCCESSFULLY
===================================================

""")



16_rag_chatbot.ipynb COMPLETED SUCCESSFULLY


